# Get a real breast cancer dataset (GSE42568) for our app

This notebook downloads the public GEO dataset **GSE42568** (104 breast cancer and 17 normal breast tissue samples, Affymetrix microarray, log2 values), converts it into the CSV format of our app, and saves it as `breast_cancer_gse42568.csv`.

**Steps:** run all cells (Runtime, Run all). At the end the CSV downloads to your computer. Then upload it to the GitHub repository (Add file, Upload files). The app shows a new data source called *Breast cancer (GSE42568, real data)*.

Group: Hira Afzal (SP23-BSI-020), Hamna Nasim (SP23-BSI-017), Namra Munir Khokhar (SP23-BSI-040)

## Step 1: Install the GEO reader

In [1]:
!pip install -q GEOparse

## Step 2: Helper functions (labels and probe to gene)

In [2]:
import numpy as np
import pandas as pd

APOPTOSIS = ["TP53","BAX","BAK1","BCL2","BCL2L1","MCL1","XIAP","BIRC5","BIRC2","BIRC3","CASP3","CASP6","CASP7","CASP8","CASP9",
             "BID","BBC3","FAS","BAD","BCL2L11","APAF1","CYCS","DIABLO","PMAIP1","FADD","MYC","EGFR","CDKN1A","VEGFA"]


def label_from_metadata(meta):
    """Return 'healthy', 'cancer' or 'unknown' from the sample description."""
    parts = []
    for key in ("title", "source_name_ch1", "characteristics_ch1"):
        v = meta.get(key, [])
        parts.append(" ".join(v) if isinstance(v, list) else str(v))
    text = " ".join(parts).lower()
    if "normal" in text or "healthy" in text:
        return "healthy"
    if any(k in text for k in ("cancer", "tumor", "tumour", "carcinoma")):
        return "cancer"
    return "unknown"


def collapse_probes(expr, ann, sym_col):
    """expr: probes x samples. ann: table with columns ID and a gene symbol column.
    Keeps one probe per gene (the probe with the highest mean expression)."""
    ann = ann[["ID", sym_col]].dropna().copy()
    ann["ID"] = ann["ID"].astype(str)
    ann[sym_col] = ann[sym_col].astype(str).str.split(" /// ").str[0].str.strip()
    ann = ann[(ann[sym_col] != "") & (ann[sym_col] != "nan")]
    probe2gene = dict(zip(ann["ID"], ann[sym_col]))
    expr = expr.copy()
    expr.index = expr.index.astype(str)
    expr = expr.loc[[p for p in expr.index if p in probe2gene]]
    tmp = expr.copy()
    tmp["gene"] = [probe2gene[p] for p in expr.index]
    tmp["_mean"] = expr.mean(axis=1).to_numpy()
    tmp = tmp.sort_values("_mean", ascending=False).drop_duplicates("gene")
    return tmp.drop(columns=["_mean"]).set_index("gene")


## Step 3: Download the dataset from GEO

In [3]:
import GEOparse

gse = GEOparse.get_GEO(geo="GSE42568", destdir="./geo_cache", silent=True)
print(gse.metadata["title"])
print("Samples:", len(gse.gsms))

/usr/local/lib/python3.13/dist-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")


['Breast Cancer Gene Expression Analysis']
Samples: 121


## Step 4: Labels (cancer or healthy) from the sample descriptions

In [4]:
labels = {gsm_id: label_from_metadata(gsm.metadata) for gsm_id, gsm in gse.gsms.items()}
print(pd.Series(labels).value_counts())
# Expected: cancer 104 and healthy 17. If the numbers are different, look at this:
first = next(iter(gse.gsms.values()))
print({k: first.metadata.get(k) for k in ("title", "source_name_ch1", "characteristics_ch1")})

cancer     104
healthy     17
Name: count, dtype: int64
{'title': ['Normal breast, N1_15_12_04'], 'source_name_ch1': ['Breast tissue, normal'], 'characteristics_ch1': ['tissue: normal breast', 'age: NA', 'er_status: NA', 'size: NA', 'grade: NA', 'lymph node status: NA', 'relapse free survival time_days: NA', 'relapse free survival event: NA', 'overall survival time_days: NA', 'overall survival event: NA']}


## Step 5: Expression table and probe to gene symbol

In [5]:
expr = gse.pivot_samples("VALUE")                    # probes x samples
platform = list(gse.gpls.values())[0]
ann = platform.table
sym_col = next(c for c in ann.columns if c.lower() in ("gene symbol", "gene_symbol", "symbol"))
gene_expr = collapse_probes(expr, ann, sym_col)         # genes x samples
print("Genes:", gene_expr.shape[0], "| Samples:", gene_expr.shape[1])
print("Value range:", round(float(gene_expr.min().min()), 2), "to", round(float(gene_expr.max().max()), 2))

Genes: 22880 | Samples: 121
Value range: 2.31 to 16.05


## Step 6: Check the scale
Microarray values from GEO are usually already log2 (roughly 2 to 15). If the maximum is above 30, the values are not log2 yet and we convert them.

In [6]:
if float(gene_expr.max().max()) > 30:
    gene_expr = np.log2(gene_expr.clip(lower=0) + 1)
    print("Converted to log2")
else:
    print("Values are already log2")

Values are already log2


## Step 7: Keep the most variable genes (plus apoptosis genes) and save

We keep 5000 genes so that the file stays small. The apoptosis genes are always kept.

In [7]:
N_TOP = 5000
variable = gene_expr.var(axis=1).sort_values(ascending=False).index[:N_TOP]
keep = list(dict.fromkeys(list(variable) + [g for g in APOPTOSIS if g in gene_expr.index]))

out = gene_expr.loc[keep].T                              # samples x genes
out["label"] = [labels[s] for s in out.index]
out = out[out["label"].isin(["healthy", "cancer"])]
out.index.name = "sample"
print(out["label"].value_counts())
print("Apoptosis genes found:", [g for g in APOPTOSIS if g in out.columns])
out.to_csv("breast_cancer_gse42568.csv")
print("Saved", out.shape)

label
cancer     104
healthy     17
Name: count, dtype: int64
Apoptosis genes found: ['TP53', 'BAX', 'BAK1', 'BCL2', 'BCL2L1', 'MCL1', 'XIAP', 'BIRC5', 'BIRC2', 'BIRC3', 'CASP3', 'CASP6', 'CASP7', 'CASP8', 'CASP9', 'BID', 'BBC3', 'FAS', 'BAD', 'BCL2L11', 'APAF1', 'CYCS', 'DIABLO', 'PMAIP1', 'FADD', 'MYC', 'EGFR', 'CDKN1A', 'VEGFA']
Saved (121, 5020)


## Step 8: Download the CSV to your computer

In [8]:
from google.colab import files
files.download("breast_cancer_gse42568.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Next
1. Upload `breast_cancer_gse42568.csv` to the GitHub repository.
2. Open the app and choose **Breast cancer (GSE42568, real data)**. Control = healthy, Case = cancer.
3. Look at DEGs, heatmap, apoptosis genes and prediction.

Note: this dataset has no mutation data, so the Mutations tab stays empty.